**CANCER THROUGH TIME** - A ML project by Rishita Arun.

In [ ]:
# importing libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVC
from sklearn.linear_model import LinearRegression

from sklearn.metrics import accuracy_score, mean_squared_error

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

In [ ]:
# styling

In [ ]:
SURFACE       = "#fcfcfb"
INK_PRIMARY   = "#0b0b0b"
INK_SECONDARY = "#52514e"
INK_MUTED     = "#898781"
GRIDLINE      = "#e1e0d9"
BORDER        = "rgba(11,11,11,0.10)"

ACCENT_BLUE   = "#2a78d6"
ACCENT_ORANGE = "#eb6834"
ACCENT_AQUA   = "#1baf7a"
ACCENT_YELLOW = "#eda100"
ACCENT_GOOD   = "#0ca30c"

MODEL_COLORS = [ACCENT_BLUE, ACCENT_ORANGE, ACCENT_AQUA, ACCENT_YELLOW]


def style_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(GRIDLINE)
    ax.spines["bottom"].set_color(GRIDLINE)
    ax.tick_params(colors=INK_MUTED)
    ax.yaxis.grid(True, color=GRIDLINE, linewidth=0.8)
    ax.set_axisbelow(True)

In [ ]:
# uploading CSV

In [ ]:
print("Upload cancer_through_time.csv")
uploaded = files.upload()
combined_filename = list(uploaded.keys())[0]

language_df = pd.read_csv(combined_filename)

print("\nDataset Loaded Successfully!")
display(language_df.head())

In [ ]:
# cleaning the data

In [ ]:
language_df = language_df.dropna(subset=["phrase", "tone_label"])
print("Language rows:", len(language_df))

In [ ]:
# records

In [ ]:
economics_records = [
    (1976, 915, 48.9, "actual"), (1980, 1000, 51.3, "interpolated"),
    (1985, 1184, 54.3, "interpolated"), (1990, 1664, 57.3, "interpolated"),
    (1995, 2135, 60.2, "interpolated"), (2000, 3332, 63.2, "interpolated"),
    (2001, 3754, 63.8, "interpolated"), (2002, 4181, 64.4, "interpolated"),
    (2003, 4592, 65.0, "interpolated"), (2004, 4739, 65.6, "interpolated"),
    (2005, 4825, 66.2, "interpolated"), (2006, 4793, 66.8, "interpolated"),
    (2007, 4798, 67.4, "interpolated"), (2008, 4831, 68.0, "interpolated"),
    (2009, 4969, 68.6, "interpolated"), (2010, 5103, 69.2, "actual"),
    (2011, 5059, 69.7, "interpolated"), (2012, 5072, 70.1, "interpolated"),
    (2013, 4807, 70.6, "interpolated"), (2014, 4923, 71.1, "interpolated"),
    (2015, 4950, 71.6, "interpolated"), (2016, 5215, 72.0, "interpolated"),
    (2017, 5689, 72.5, "actual"), (2018, 5965, 73.0, "extrapolated"),
    (2019, 6144, 73.4, "extrapolated"), (2020, 6440, 73.9, "extrapolated"),
    (2025, 7224, 76.3, "extrapolated"),
]

economics_df = pd.DataFrame(
    economics_records,
    columns=["year", "funding_million_usd", "survival_rate_percent", "survival_data_type"]
)

print("Economics rows:", len(economics_df))
display(economics_df.head())

In [ ]:
# training and testing the data

In [ ]:
X_text = language_df["phrase"]
y_tone = language_df["tone_label"]

X_train_text, X_test_text, y_train_tone, y_test_tone = train_test_split(
    X_text,
    y_tone,
    test_size=0.25,
    random_state=42,
    stratify=y_tone
)

eval_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=300
)

X_train_vec = eval_vectorizer.fit_transform(X_train_text)
X_test_vec = eval_vectorizer.transform(X_test_text)

In [ ]:
# comparing the 4 classifiers

In [ ]:
classifiers = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=3),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(probability=True, random_state=42)
}

tone_accuracies = {}

for name, model in classifiers.items():
    model.fit(X_train_vec, y_train_tone)
    pred = model.predict(X_test_vec)
    acc = accuracy_score(y_test_tone, pred)
    tone_accuracies[name] = acc
    print(f"{name} Accuracy: {acc:.2f}")

best_tone_model_name = max(tone_accuracies, key=tone_accuracies.get)

print("\n================================")
print("BEST LANGUAGE MODEL:", best_tone_model_name)
print(f"BEST ACCURACY: {tone_accuracies[best_tone_model_name]:.2f}")
print("================================")
print("\nNote: this dataset is small (a few dozen phrases per era), so accuracy")
print("in the 50-65% range is expected and honest - not a bug. A tiny, hand-built")
print("text dataset like this is a good example of why real NLP projects need")
print("much larger corpora to get reliably high accuracy.")

In [ ]:
# plotting the comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
fig.patch.set_facecolor(SURFACE)
ax.set_facecolor(SURFACE)

names = list(tone_accuracies.keys())
values = list(tone_accuracies.values())

bars = ax.bar(names, values, color=MODEL_COLORS, width=0.55)

ax.set_title("Language Classifier — Accuracy by Model", color=INK_PRIMARY, fontsize=13, loc="left")
ax.set_ylabel("Accuracy", color=INK_SECONDARY)
ax.set_ylim(0, 1.1)
style_axes(ax)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{val:.2f}", ha="center", color=INK_PRIMARY, fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# cross validating the data to give steadier result

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv_vectorizer = TfidfVectorizer(stop_words="english", max_features=300)
X_all_vec_cv = cv_vectorizer.fit_transform(X_text)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("5-fold cross-validation accuracy (mean ± std):\n")
for name, model in classifiers.items():
    scores = cross_val_score(model, X_all_vec_cv, y_tone, cv=skf, scoring="accuracy")
    print(f"{name:15s} {scores.mean():.2f} ± {scores.std():.2f}   (folds: {np.round(scores, 2)})")

print("\nIf a model's rank changes here versus the single-split chart above,")
print("that's a sign the single split was lucky/unlucky rather than a real")
print("difference in model quality.")

In [ ]:
# refitting

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english", max_features=300)
X_all_vec = vectorizer.fit_transform(X_text)

deploy_classifiers = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=3),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(probability=True, random_state=42)
}

best_tone_model = deploy_classifiers[best_tone_model_name]
best_tone_model.fit(X_all_vec, y_tone)

print("Deployed model:", best_tone_model_name, "(refit on all", len(X_text), "phrases)")

In [ ]:
# training and testing economics data

In [ ]:
X_econ = economics_df[["year", "funding_million_usd"]]
y_survival = economics_df["survival_rate_percent"]

X_train_econ, X_test_econ, y_train_surv, y_test_surv = train_test_split(
    X_econ,
    y_survival,
    test_size=0.25,
    random_state=42
)

In [ ]:
# comparing the 4 regressors

In [ ]:
regressors = {
    "Decision Tree": DecisionTreeRegressor(max_depth=4, random_state=42),
    "KNN": KNeighborsRegressor(n_neighbors=3),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=4, random_state=42),
    "Linear Regression": LinearRegression()
}

survival_rmse = {}
trained_regressors = {}

for name, model in regressors.items():
    model.fit(X_train_econ, y_train_surv)
    pred = model.predict(X_test_econ)
    rmse = np.sqrt(mean_squared_error(y_test_surv, pred))
    survival_rmse[name] = rmse
    trained_regressors[name] = model
    print(f"{name} RMSE: {rmse:.3f}")

best_survival_model_name = min(survival_rmse, key=survival_rmse.get)
best_survival_model = trained_regressors[best_survival_model_name]

print("\n================================")
print("BEST SURVIVAL MODEL:", best_survival_model_name)
print(f"BEST RMSE: {survival_rmse[best_survival_model_name]:.3f}")
print("================================")

In [ ]:
# plotting the regressors

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
fig.patch.set_facecolor(SURFACE)
ax.set_facecolor(SURFACE)

names = list(survival_rmse.keys())
values = list(survival_rmse.values())

bars = ax.bar(names, values, color=MODEL_COLORS, width=0.55)

ax.set_title("Survival Regressor — RMSE by Model (lower is better)", color=INK_PRIMARY, fontsize=13, loc="left")
ax.set_ylabel("RMSE (percentage points)", color=INK_SECONDARY)
style_axes(ax)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{val:.2f}", ha="center", color=INK_PRIMARY, fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# cross validation

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("5-fold cross-validation RMSE (mean ± std, lower is better):\n")
for name, model in regressors.items():
    neg_mse_scores = cross_val_score(
        model, X_econ, y_survival, cv=kf, scoring="neg_mean_squared_error"
    )
    rmse_scores = np.sqrt(-neg_mse_scores)
    print(f"{name:18s} {rmse_scores.mean():.2f} ± {rmse_scores.std():.2f}   (folds: {np.round(rmse_scores, 2)})")

print("\nAs with Part A, compare ranks here to the single-split chart above —")
print("with only 27 rows, a close single-split result can flip either way.")

In [ ]:
# Eras

In [ ]:
ERA_DECADE_LABEL = dict(zip(language_df["era"], language_df["decade_label"]))

# One representative ("hero") phrase per era, shown in the explorer card.
# Chosen to be clearly, correctly classified by the deployed model.
HERO_PHRASE = {
    "1970s": "The word itself was rarely spoken aloud in polite company.",
    "1980s-1990s": "Early detection through mammography can save lives.",
    "1990s-2000s": "Doctors now personalize treatment based on tumor genetics.",
    "2010s": "She calls herself a survivor, not a victim.",
    "2020s": "The Cancer Moonshot aims to cut death rates in half within a generation."
}

known_years = economics_df["year"].values
known_funding = economics_df["funding_million_usd"].values


def era_for_year(year):
    if year < 1980:
        return "1970s"
    if year < 2000:
        return "1980s-1990s"
    if year < 2010:
        return "1990s-2000s"
    if year < 2020:
        return "2010s"
    return "2020s"


def funding_for_year(year):
    # Numeric interpolation between known, real yearly funding figures
    return float(np.interp(year, known_years, known_funding))


def predict_survival(year, funding):
    row = pd.DataFrame([[year, funding]], columns=["year", "funding_million_usd"])
    return float(best_survival_model.predict(row)[0])


def classify_tone(phrase):
    vec = vectorizer.transform([phrase])
    pred_label = best_tone_model.predict(vec)[0]
    proba = best_tone_model.predict_proba(vec)[0]
    return pred_label, proba.max()

def data_provenance_for_year(year):
    """
    Looks up whether `year` is one of the known rows in economics_df, and
    if so, what kind of number its survival rate is: actual SEER data,
    or an interpolated/extrapolated estimate. For a year that isn't an
    exact row (the slider allows any year 1976-2025), the number shown
    comes from the trained regressor, not the lookup table at all.
    """
    match = economics_df.loc[economics_df["year"] == year, "survival_data_type"]
    if len(match) == 0:
        return "model prediction"
    kind = match.iloc[0]
    if kind == "actual":
        return "actual SEER data"
    if kind == "interpolated":
        return "interpolated estimate"
    return "extrapolated estimate"


PROVENANCE_COLOR = {
    "actual SEER data": ACCENT_GOOD,
    "interpolated estimate": ACCENT_YELLOW,
    "extrapolated estimate": ACCENT_ORANGE,
    "model prediction": INK_MUTED,
}

In [ ]:
# Year Explorer

In [ ]:
year_slider = widgets.IntSlider(
    value=1976,
    min=1976,
    max=2025,
    step=1,
    description="Year:",
    continuous_update=False,
    style={"description_width": "50px"},
    layout=widgets.Layout(width="600px")
)

output = widgets.Output()


def render_explorer(change=None):
    with output:
        clear_output()

        year = year_slider.value
        era = era_for_year(year)
        decade_label = ERA_DECADE_LABEL.get(era, era)

        phrase = HERO_PHRASE[era]
        tone_label, confidence = classify_tone(phrase)

        funding = funding_for_year(year)
        survival = predict_survival(year, funding)
        provenance_label = data_provenance_for_year(year)
        provenance_color = PROVENANCE_COLOR[provenance_label]

        display(HTML(f"""
        <div style="
            background-color:{SURFACE};
            background-image:
                linear-gradient({GRIDLINE}4d 1px, transparent 1px),
                linear-gradient(90deg, {GRIDLINE}4d 1px, transparent 1px);
            background-size: 24px 24px;
            border:1px solid {BORDER};
            border-radius:12px;
            padding:28px;
            font-family:system-ui,-apple-system,'Segoe UI',sans-serif;
            color:{INK_PRIMARY};
        ">
            <div style="border-bottom:1px solid {GRIDLINE}; padding-bottom:14px; margin-bottom:20px;">
                <div style="color:{INK_MUTED}; font-size:13px; letter-spacing:0.04em; text-transform:uppercase;">
                    {year}
                </div>
                <div style="font-size:20px; font-weight:600;">
                    {decade_label}
                </div>
            </div>

            <div style="display:flex; gap:16px; flex-wrap:wrap;">

                <div style="flex:1; min-width:240px; border:1px solid {BORDER}; border-radius:10px; padding:18px;">
                    <div style="color:{INK_MUTED}; font-size:12px; text-transform:uppercase; letter-spacing:0.04em;">Language</div>
                    <div style="font-size:15px; margin:10px 0; line-height:1.5;">&ldquo;{phrase}&rdquo;</div>
                    <div style="color:{ACCENT_BLUE}; font-size:13px; font-weight:600;">
                        {tone_label.replace('_', ' ').title()} &middot; {confidence * 100:.0f}% model confidence
                    </div>
                </div>

                <div style="flex:1; min-width:160px; border:1px solid {BORDER}; border-radius:10px; padding:18px;">
                    <div style="color:{INK_MUTED}; font-size:12px; text-transform:uppercase; letter-spacing:0.04em;">Research Funding</div>
                    <div style="font-size:28px; font-weight:600; margin-top:10px;">${funding:,.0f}M</div>
                    <div style="color:{INK_SECONDARY}; font-size:12px; margin-top:4px;">NCI annual budget</div>
                </div>

                <div style="flex:1; min-width:160px; border:1px solid {BORDER}; border-radius:10px; padding:18px;">
                    <div style="color:{INK_MUTED}; font-size:12px; text-transform:uppercase; letter-spacing:0.04em;">5-Year Survival</div>
                    <div style="font-size:28px; font-weight:600; margin-top:10px; color:{ACCENT_GOOD};">{survival:.1f}%</div>
                    <div style="color:{INK_SECONDARY}; font-size:12px; margin-top:4px;">predicted by {best_survival_model_name}</div>
                    <div style="display:inline-block; margin-top:10px; padding:3px 10px; border-radius:20px; background:{provenance_color}1a; color:{provenance_color}; font-size:11px; font-weight:600;">
                        {provenance_label}
                    </div>
                </div>

            </div>
        </div>
        """))

        fig, ax = plt.subplots(figsize=(8, 3.2))
        fig.patch.set_facecolor(SURFACE)
        ax.set_facecolor(SURFACE)

        ax.plot(economics_df["year"], economics_df["survival_rate_percent"],
                color=ACCENT_BLUE, linewidth=2)
        ax.axvline(year, color=ACCENT_ORANGE, linewidth=1.5, linestyle="--")
        ax.scatter([year], [survival], color=ACCENT_ORANGE, zorder=5, s=40)

        ax.set_title("5-Year Survival Rate, 1976-2025", color=INK_PRIMARY, fontsize=12, loc="left")
        ax.set_ylabel("%", color=INK_SECONDARY)
        style_axes(ax)

        plt.tight_layout()
        plt.show()

year_slider.observe(render_explorer, names="value")

In [ ]:
# Guess the Era

In [ ]:
TONE_TO_ERA = dict(zip(language_df["tone_label"], language_df["era"]))

sentence_box = widgets.Text(
    placeholder="Type a sentence about cancer, or paste one from the CSV...",
    description="Sentence:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="600px")
)

guess_button = widgets.Button(
    description="Guess the Era",
    button_style="success",
    layout=widgets.Layout(width="160px", height="36px")
)

guess_output = widgets.Output()


def guess_era(b):
    with guess_output:
        clear_output()

        sentence = sentence_box.value.strip()
        if sentence == "":
            print("Type a sentence first.")
            return

        vec = vectorizer.transform([sentence])
        if vec.nnz == 0:
            display(HTML(f"""
            <div style="background:{SURFACE}; border:1px solid {BORDER};
                        border-radius:10px; padding:18px; font-family:system-ui,sans-serif;
                        color:{INK_PRIMARY};">
                That doesn't sound like it's about cancer or health at all.
                Try another sentence?
            </div>
            """))
            return

        tone_label, confidence = classify_tone(sentence)
        if confidence < 0.40:
            display(HTML(f"""
            <div style="background:{SURFACE}; border:1px solid {BORDER};
                        border-radius:10px; padding:18px; font-family:system-ui,sans-serif;
                        color:{INK_PRIMARY};">
                I'm not confident that's about cancer or health.
                Try another sentence?
            </div>
            """))
            return

        era = TONE_TO_ERA[tone_label]
        decade_label = ERA_DECADE_LABEL[era]

        display(HTML(f"""
        <div style="background:{SURFACE}; border:1px solid {BORDER};
                    border-radius:10px; padding:18px; font-family:system-ui,sans-serif;
                    color:{INK_PRIMARY};">
            <span style="font-size:16px;">That sounds like <b>{decade_label}</b></span><br>
            <span style="color:{INK_SECONDARY}; font-size:13px;">{confidence * 100:.0f}% model confidence</span>
        </div>
        """))


guess_button.on_click(guess_era)

In [ ]:
# Display

In [ ]:
combined_panel = widgets.VBox([
    year_slider,
    output,
    widgets.HTML(f"<hr style='margin:24px 0; border:none; border-top:1px solid {GRIDLINE};'>"),
    sentence_box,
    guess_button,
    guess_output
])

display(combined_panel)
render_explorer()